In [1]:
import os
import sys

project_root = '/home/jovyan/project_10x'

sys.path.append(os.path.join(project_root, 'src'))

from utils import *
from sentiment_extraction_llama_10k import *

import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

/tmp/ipykernel_511689/2410463052.py:17: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Доступно GPU: {torch.cuda.device_count()}")
print(f"Имена устройств: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

Доступно GPU: 2
Имена устройств: ['NVIDIA A100-SXM4-80GB', 'NVIDIA A100 80GB PCIe']


In [6]:
clean_mem()

In [7]:
sys.version

'3.11.11 | packaged by conda-forge | (main, Mar  3 2025, 20:43:55) [GCC 13.3.0]'

# Data

In [15]:
# report_path = '/content/drive/MyDrive/10-X/edgar_data/'
# reports = pd.read_csv(os.path.join(report_path, '10k_filtered.tsv.gz'), sep='\t')

report_path = '/home/jovyan/datavol-2/'
reports = pd.read_csv(os.path.join(report_path, 'reports_100sample.tsv'), sep='\t')
reports.columns = ['index', 'CIK', 'FILING_DATE', 'ACC_NUM']
reports = reports.set_index('index')
reports.tail()

,CIK,FILING_DATE,ACC_NUM
index,,,
866,1566610,20230417,0001493152-23-012571
955,1370946,20190220,0001370946-19-000012
39,1475922,20180226,0001564590-18-003151
640,20286,20220224,0000020286-22-000012
732,1433195,20210301,0001433195-21-000024


In [16]:
reports.shape

(109, 3)

In [17]:
items_path = '/home/jovyan/datavol-2/sample_item7/'
dataset = Dataset10x(reports, items_path)

In [18]:
len(dataset)

109

In [19]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn_item7)
idx, texts = next(iter(dataloader))

# Model - Llama

In [20]:
version = '3.1'
# Context length = 128k

if version=='3':
    model_id = "meta-llama/Meta-Llama-3-8B"
elif version=='3.1':
#     model_id = "meta-llama/Meta-Llama-3.1-8B"
    model_id = '/home/jovyan/models-2/Meta-Llama-3.1-8B'
elif version=='3.2':
    model_id = "meta-llama/Llama-3.2-3B"
elif version=='3.2-1b':
    model_id = "meta-llama/Llama-3.2-1B"
else:
    raise Exception("Set correct version")

print(f"Model {version=} -- {model_id=}")
logging.set_verbosity_error()

Model version='3.1' -- model_id='/home/jovyan/models-2/Meta-Llama-3.1-8B'


In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir="hf_cache")
if not tokenizer.pad_token:
    print("Adding pad_token")
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
vocab = set([*tokenizer.get_vocab()])

model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, cache_dir="hf_cache"
)
# .cuda()

# if torch.cuda.device_count() > 1:
#     model = torch.nn.DataParallel(model, device_ids=list(range(8)))

# model = torch.nn.DataParallel(model)
# model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Используем {torch.cuda.device_count()} GPU")
    model = torch.nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.cuda() 

model.name = 'Llama3_1'

Adding pad_token


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  8.32it/s]


Используем 2 GPU


In [22]:
clean_mem()
item_name = 'item7'

In [23]:
model.device_ids

[0, 1]

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Main prompts strategies:

1) Financial health

2) Revenues;

3) Profit;

4) 5 types of risks;

5) Industry -- for validation.

In [25]:
@dataclass(frozen=True)
class Prompt_Strategy:
    name: str
    verbalizer: dict
    prompt: Callable
    top_p: float = 0.9

In [26]:
clean_mem()

# global inputs
# del inputs
gc.collect()
torch.cuda.empty_cache()

In [27]:
q95 = 179501
q99 = 292229

## Financial risk

In [28]:
def get_financial_risk_prompt(item_text: str) -> str:
#     beginning = f"This is a part of a financial report of the company, here it is:"
#     ending = "The main risk facing the company is the"
#     ending = "If I had to name the most significant risk facing the company I would mention"
#     ending = "In a single word, the main risk facing our company is the risk of"
#     ending = "List of risks facing the company: 1."
#     ending = "Our risk exposure regarding the forthcoming election is"

#     ending = "In summary, I would say that the level of strategic risk facing the company is"
#     ending = "In summary, I would say that the level of operational risk facing the company is"
    ending = "In summary, I would say that the level of financial risk facing the company is"
    prompt = f"{item_text}\n{ending}"
    return prompt

In [29]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn_item7)
# idx, texts = next(iter(dataloader))
len(dataloader)

55

In [32]:
clean_mem()

In [31]:
words_risks = []

for idx, texts in notebook_tqdm(dataloader):
    
    res_risk = fill_prompt_batch(
        texts, get_financial_risk_prompt,
        tokenizer=tokenizer, model=model,
        top_p=0.9, text_length=q99,
        device=device
    )
    for i in range(len(res_risk)):
        words_risks.extend(list(res_risk[i].keys()))
        
    clean_mem()
    time.sleep(1)

words_risk_set = set([w.strip().lower() for w in words_risks])

print(len(words_risk_set))
print(words_risk_set)

  0%|          | 0/55 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [34]:
risk_verb = {
    "positive": set([
                     'balanced', 'controlled', 'controlled:', 'declined',
                     'declining', 'declining:', 'decreased', 'decreasing', 'decreasing:',
                     'governed', 'insignificant', 'insignificant:', 'limited', 'limited,',
                     'low', 'lower', 'lower,', 'lower:', 'manageable', 'managed', 'managed:',
                     'minimal', 'minor', 'mitig', 'mitig:', 'mitigated',
                     'moder', 'moderate', 'moderated', 'moderated:', 'moderately',
                     'moderately,', 'modest', 'modest,', 'modest:', 'negligible',
                     'negligible:', 'reduced', 'reduced,', 'reduced:', 'small', 'small:',
                     'slight', 'reducing', 
                    ]),
    
    "negative": set([
                     'considerable', 'considerable,', 'considerable:', 'considerably', 'extensive',
                     'exacerbated', 'extraordinarily', 'extraordinary', 'extreme', 'extreme:', 'elevated',
                     'great', 'greater', 'greater,', 'growing', 'growing:',
                     'heightened', 'heightened:', 'high', 'higher', 'highest', 
                     'huge', 'immense', 'increased', 
                     'increased,', 'increased:', 'increasing', 'large', 'rising',
                     'serious', 'severe', 'severe:', 'significant', 'significant,',
                     'significant:', 'significantly', 'significantly,', 'unprecedented', 
                     'substantial', 'substantial,', 'sufficient', 'tremendous', 'unacceptable',
                    ])
}

risk_prompt = get_financial_risk_prompt
risk_strategy = Prompt_Strategy('financial_risk', risk_verb, risk_prompt)

print(len(risk_verb['positive']), len(risk_verb['negative']))

44 43


## Whole dataset

In [35]:
report_path = '/home/jovyan/datavol-2/'
reports = pd.read_csv(os.path.join(report_path, '10k_filtered.tsv.gz'), sep='\t')
reports = reports[['CIK', 'FILING_DATE', 'ACC_NUM']]
reports.shape

(50260, 3)

In [36]:
items_path = '/home/jovyan/datavol-2/items/item7_files'
dataset = Dataset10x(reports, items_path)

In [37]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn_item7)
len(dataloader)

25130

In [38]:
q95 = 179501
q99 = 292229

results = []

In [ ]:
stats_sent = gather_stats(risk_strategy, results=results, tokenizer=tokenizer, model=model,
                          data=dataloader, verbose=False, text_length=q95, 
                          save_path="/home/jovyan/datavol-2/financial_risk_sentiment/",
                          save_interval=2000, resume=True, max_retries=3)

Starting from beginning


  4%|▍         | 1051/25130 [2:15:44<27:04:17,  4.05s/it] 

Saved CSV: /home/jovyan/datavol-2/financial_risk_sentiment/results_20250828_172938_2001_reports.csv


  4%|▍         | 1115/25130 [2:24:51<99:58:47, 14.99s/it] 